# SC4021 Assignment Task 1 Crawling

This notebook builds a corpus of smartphone-related user opinions from Hacker News using the Algolia API and the official HN Firebase API. Target devices include the iPhone 15, iPhone 15 Pro, Galaxy S24, Galaxy S23, Pixel 8, Pixel 7, OnePlus 12, and Xiaomi 14. The workflow is as follows:
 1. Crawls Hacker News stories and comments matching device-related queries (e.g. "Pixel 8 battery", "Galaxy S23 review") for the selected smartphones via Algolia, then enriches results with the official Firebase API (deleted, descendants, score, item_type).
 2. Processes the raw data by removing HTML, deduplicating entries, applying quality filters, and sanitizing text for compatibility with Excel exports.
 3. Saves results to `corpus_full.xlsx` (containing all metadata) and `eval.xlsx` (containing text and annotation label columns).

## 1. Importing Libraries

In [2]:
# !pip -q install pandas requests openpyxl

In [ ]:
import html
import re
import time
import json
import random
import hashlib
import unicodedata
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

## 2. Configuration

In [ ]:
# Target devices to search for in HN comments and stories
DEVICES = [
    "iPhone 15",
    "iPhone 15 Pro",
    "Galaxy S24",
    "Galaxy S23",
    "Pixel 8",
    "Pixel 7",
    "OnePlus 12",
    "Xiaomi 14"
]

# Algolia API endpoint for searching by date
ALGOLIA_SEARCH_BY_DATE = "https://hn.algolia.com/api/v1/search_by_date"

# Hacker News Firebase API endpoint base URL
HN_FIREBASE_BASE = "https://hacker-news.firebaseio.com/v0"

# Firebase API: delay between requests
FIREBASE_SLEEP = 0.15

## 3. Crawling & Initial Processing

- We crawl Hacker News via the Algolia API using time-sliced queries. For each device, we search for "device name", "review", "battery", "camera", "problems", "worth it" etc as defined in `build_hn_queries()` below. We also add comparisons like "iPhone 15 vs Galaxy S24".

- The same HN post can match multiple queries (e.g. a comment about "iPhone 15 battery" appears in both "iPhone 15" and "iPhone 15 battery"). Therefore, we assign each document a unique `doc_id` and deduplicate to keep one row per unique post.

- After crawling, we further enrich the results by using the official HN Firebase API (`/v0/item/{id}.json`) to add attributes like:
     - `deleted` (whether the post/comment was deleted)
     - `descendants` (number of direct replies for comments, or total descendants for stories)
     - `score` (number of upvotes; for stories only, comments have 0)

In [ ]:
# --- Crawl utilities ---
# Normalize and clean up a string (strip, squeeze spaces)
def normalize_text(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s

# Count number of words in a string
def word_count(s: str) -> int:
    return len((s or "").split())

# Generate a unique doc_id from document source, id, and text snippet
def make_doc_id(source: str, object_id: str, text: str) -> str:
    base = f"{source}|{object_id}|{(text or '')[:500]}"
    return hashlib.sha1(base.encode("utf-8", errors="ignore")).hexdigest()

# Remove HTML tags and unescape entities from a string
def clean_html(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"<.*?>", " ", s)
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

# --- Algolia API ---
# Convert a datetime object to a unix timestamp (seconds since epoch, UTC)
def dt_to_unix(dt: datetime) -> int:
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp())

# Query Algolia API with GET and error handling/retries
def algolia_get(url: str, params: dict, retries: int = 5, headers: dict = None):
    hdrs = headers or {"User-Agent": "Mozilla/5.0 (SC4021-opinion-crawler/1.0)"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=hdrs, timeout=25)
            r.raise_for_status()
            return r.json()
        except (requests.HTTPError, requests.RequestException) as e:
            sleep_s = (2 ** attempt) + random.random()
            print("Req error:", e, "| sleeping", round(sleep_s, 2), "s")
            time.sleep(sleep_s)
    return None

# Build HN queries for device-related searches
def build_hn_queries(devices: list[str]) -> list[str]:
    templates = ["{d}", "{d} review", "{d} long term review", "{d} worth it",
                 "{d} should I buy", "{d} upgrade", "{d} problems", "{d} issues",
                 "{d} battery", "{d} camera", "{d} overheating", "{d} performance",
                 "{d} regret", "{d} love", "{d} hate"]
    comparisons = []
    for i in range(len(devices)):
        for j in range(i+1, min(i+3, len(devices))):
            comparisons += [f"{devices[i]} vs {devices[j]}", f"{devices[i]} or {devices[j]}"]
    qs = [t.format(d=d) for d in devices for t in templates] + comparisons
    out, seen = [], set()
    for q in qs:
        q = q.strip()
        if q and q not in seen:
            out.append(q)
            seen.add(q)
    return out

# Crawl Hacker News via Algolia API with time-sliced queries
def crawl_hn_time_sliced(queries, start_date, end_date, slice_days=30, hits_per_page=200,
    max_pages_per_slice=5, tags="(comment,story)", min_chars=80, sleep_s=0.5):
    rows = []
    start_date = start_date.replace(tzinfo=timezone.utc)
    end_date = end_date.replace(tzinfo=timezone.utc)
    for qi, q in enumerate(queries, start=1):
        print(f"\n[{qi}/{len(queries)}] HN query: {q}")
        win_start = start_date
        while win_start < end_date:
            win_end = min(win_start + timedelta(days=slice_days), end_date)
            a, b = dt_to_unix(win_start), dt_to_unix(win_end)
            nf = [f"created_at_i>{a}", f"created_at_i<{b}"]
            for page in range(max_pages_per_slice):
                params = {"query": q, "tags": tags, "numericFilters": ",".join(nf),
                          "hitsPerPage": hits_per_page, "page": page}
                # Query Algolia API
                data = algolia_get(ALGOLIA_SEARCH_BY_DATE, params=params)
                if not data:
                    break
                hits = data.get("hits", []) or []
                if not hits:
                    break
                for h in hits:
                    title_raw = h.get("title") or h.get("story_title") or "" # title of story or comment
                    body = h.get("comment_text") or h.get("story_text") or "" # body of story or comment
                    title_clean = clean_html(title_raw)
                    body_clean = clean_html(body)
                    text = (body_clean or title_clean).strip()
                    if len(text) < min_chars: # skip if too short
                        continue
                    obj_id = str(h.get("objectID", "")) # unique ID of story or comment
                    url = h.get("url") or h.get("story_url") or "" # URL of story or comment
                    created = h.get("created_at") or "" # date/time of creation
                    author = h.get("author") or "" # author of story or comment
                    ttags = ",".join(h.get("_tags", [])) if isinstance(h.get("_tags"), list) else "" # tags of story or comment e.g. "story", "comment"
                    rows.append({"doc_id": make_doc_id("hn_algolia", obj_id, text), "object_id": obj_id,
                        "title": normalize_text(title_clean), "query": q, "created_at": created,
                        "author": author, "tags": ttags, "source_url": url, "text": normalize_text(text)})
                time.sleep(sleep_s + random.random() * 0.3)
                if page + 1 >= data.get("nbPages", 0):
                    break
            win_start = win_end
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["doc_id"])
        df = df.drop_duplicates(subset=["text"])
        df = df[df["text"].astype(str).str.len() > 0]
    return df

# --- Firebase API ---
# Get an item's JSON from the HN Firebase API
def hn_firebase_get(item_id, retries: int = 5):
    url = f"{HN_FIREBASE_BASE}/item/{item_id}.json"
    for attempt in range(retries):
        try:
            r = requests.get(url, headers={"User-Agent": "SC4021-opinion-crawler/1.0"}, timeout=15)
            # Handle rate limiting (429) by waiting and retrying
            if r.status_code == 429:
                wait = (2 ** attempt) + random.uniform(1, 3)
                print(f"  Rate limit (429) | waiting {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            data = r.json()
            if data is None:
                return {"deleted": True, "descendants": 0}
            return data
        except requests.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0.5, 2)
            if attempt < retries - 1:
                print(f"  Firebase error: {e} | retry in {wait:.1f}s")
                time.sleep(wait)
            else:
                return None
    return None

# Enrich the corpus with Firebase API data (deleted, descendants, score, item_type)
def enrich_with_firebase(df: pd.DataFrame, wait: float = 0.15) -> pd.DataFrame:
    if "object_id" not in df.columns:
        return df
    out = df.copy()
    out["deleted"] = False
    out["descendants"] = 0
    out["score"] = 0
    out["item_type"] = ""
    seen = {}
    ids = [x for x in out["object_id"].unique().tolist() if x is not None and str(x).strip()]
    n = len(ids)

    # Iterate through each unique object_id to fetch data from Firebase
    for i, oid in enumerate(ids):
        if (i + 1) % 500 == 0 or i == 0:
            print(f"  [{i+1}/{n}]") # Print status
        if (i + 1) % 1000 == 0 and i > 0:
            time.sleep(5) # Prevent rate limiting

        item = hn_firebase_get(oid) # Fetch item data from the Firebase API
        if item is None:
            seen[oid] = {"deleted": False, "descendants": 0, "score": 0, "item_type": ""}
        else:
            deleted = item.get("deleted") or item.get("dead") or False # Determine if item is deleted or dead
            desc = item.get("descendants") # Get number of descendants (comments)
            if desc is None and "kids" in item: # Fallback to counting 'kids' if not present
                desc = len(item["kids"]) if isinstance(item["kids"], list) else 0
            else:
                desc = int(desc) if desc is not None else 0
            score = int(item.get("score") or 0) # Get score (upvotes)
            item_type = item.get("type") or "" # Get type: "comment", "story"
            seen[oid] = {"deleted": deleted, "descendants": desc, "score": score, "item_type": item_type}
        time.sleep(wait)
    
    # Enrich the DataFrame with fetched fields
    for idx, oid in out["object_id"].items():
        v = seen.get(oid, {"deleted": False, "descendants": 0, "score": 0, "item_type": ""})
        out.at[idx, "deleted"] = v["deleted"]
        out.at[idx, "descendants"] = v["descendants"]
        out.at[idx, "score"] = v["score"]
        out.at[idx, "item_type"] = v["item_type"]
    n_del = out["deleted"].sum()
    if n_del > 0:
        print(f"  Found {int(n_del)} deleted items")
    return out

In [ ]:
# Run crawl
hn_queries = build_hn_queries(DEVICES)
start = datetime(2020, 1, 1, tzinfo=timezone.utc)
end = datetime.now(timezone.utc)

df_hn = crawl_hn_time_sliced(
    queries=hn_queries,
    start_date=start,
    end_date=end,
    slice_days=30,
    hits_per_page=200,
    max_pages_per_slice=5,
    tags="(comment,story)",
    min_chars=80
)

print("HN rows:", len(df_hn), "| words:", int(df_hn["text"].map(word_count).sum()))

df_hn = enrich_with_firebase(df_hn, wait=globals().get("FIREBASE_SLEEP", 0.15))

print(df_hn[["query", "author", "item_type", "deleted", "descendants", "score"]].head(5))


[1/146] HN query: iPhone 15

[2/146] HN query: iPhone 15 review

[3/146] HN query: iPhone 15 long term review

[4/146] HN query: iPhone 15 worth it

[5/146] HN query: iPhone 15 should I buy

[6/146] HN query: iPhone 15 upgrade

[7/146] HN query: iPhone 15 problems

[8/146] HN query: iPhone 15 issues

[9/146] HN query: iPhone 15 battery

[10/146] HN query: iPhone 15 camera

[11/146] HN query: iPhone 15 overheating

[12/146] HN query: iPhone 15 performance

[13/146] HN query: iPhone 15 regret

[14/146] HN query: iPhone 15 love

[15/146] HN query: iPhone 15 hate

[16/146] HN query: iPhone 15 Pro

[17/146] HN query: iPhone 15 Pro review

[18/146] HN query: iPhone 15 Pro long term review

[19/146] HN query: iPhone 15 Pro worth it

[20/146] HN query: iPhone 15 Pro should I buy

[21/146] HN query: iPhone 15 Pro upgrade

[22/146] HN query: iPhone 15 Pro problems

[23/146] HN query: iPhone 15 Pro issues

[24/146] HN query: iPhone 15 Pro battery

[25/146] HN query: iPhone 15 Pro camera

[26/146

In [28]:
# Save raw scraped data for future use
df_hn.to_parquet("hn_raw.parquet", index=False)

## 4. Data Exploration

In [ ]:
# Load raw scraped data
df_hn = pd.read_parquet("hn_raw.parquet")
df_hn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18726 entries, 0 to 18725
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   doc_id       18726 non-null  object
 1   object_id    18726 non-null  object
 2   title        18726 non-null  object
 3   query        18726 non-null  object
 4   created_at   18726 non-null  object
 5   author       18726 non-null  object
 6   tags         18726 non-null  object
 7   source_url   18726 non-null  object
 8   text         18726 non-null  object
 9   deleted      18726 non-null  bool  
 10  descendants  18726 non-null  int64 
 11  score        18726 non-null  int64 
 12  item_type    18726 non-null  object
dtypes: bool(1), int64(2), object(10)
memory usage: 1.7+ MB


In [35]:
# Use one query and a narrow date range to get a small sample
sample_query = hn_queries[0] if "hn_queries" in globals() and hn_queries else "iPhone 15"
start_ts = int(datetime(2024, 1, 1, tzinfo=timezone.utc).timestamp())
end_ts = int(datetime(2024, 1, 31, tzinfo=timezone.utc).timestamp())

params = {
    "query": sample_query,
    "tags": "(comment,story)",
    "numericFilters": f"created_at_i>{start_ts},created_at_i<{end_ts}",
    "hitsPerPage": 2,
    "page": 0
}

print("ALGOLIA API: search_by_date")
print(f"Query: {sample_query}")
data = algolia_get(ALGOLIA_SEARCH_BY_DATE, params=params)
if data:
    print(f"\nTotal results found: {data.get('nbHits', 'N/A')} | Current page: {data.get('page', 0) + 1} | Total pages: {data.get('nbPages', 0)}")    
    hits = data.get("hits", []) or []
    if hits:
        print(json.dumps(hits[0], indent=2, default=str))
    else:
        print("(no hits)")
else:
    print("(fetch failed)")

ALGOLIA API: search_by_date
Query: iPhone 15

Total results found: 116 | Current page: 1 | Total pages: 58
{
  "_highlightResult": {
    "author": {
      "matchLevel": "none",
      "matchedWords": [],
      "value": "kypro"
    },
    "comment_text": {
      "fullyHighlighted": false,
      "matchLevel": "full",
      "matchedWords": [
        "iphone",
        "15"
      ],
      "value": "I think people are overly attributing the end of low interest rates to the current job market in tech.<p>The truth is over last 10-20 years we've gone through a huge technological boom which has driven the job market in tech. In the mid 2000s almost no one knew how to code. Software engineering was more a hobby than a career choice back then. Yet, from 2005-2015 we had innovation after innovation which demanded people with coding skills.<p>This article mentions things like the launch of the <em>iPhone</em> and AWS, but there was so much more than just that. The switch from dial-up to broadband mea

In [ ]:
# Print raw Firebase API output to see what the official HN API returns
# Get one comment and one story from the corpus
story_row = df_hn[df_hn["item_type"] == "story"].head(1)
comment_row = df_hn[df_hn["item_type"] == "comment"].head(1)
story_id = story_row["object_id"].iloc[0]
comment_id = comment_row["object_id"].iloc[0]

print("FIREBASE API: STORY (id={})".format(story_id))
story = hn_firebase_get(story_id)
if story:
    print(json.dumps(story, indent=2, default=str))
else:
    print("(fetch failed)")

print("\nFIREBASE API: COMMENT (id={})".format(comment_id))
comment = hn_firebase_get(comment_id)
if comment:
    print(json.dumps(comment, indent=2, default=str))
else:
    print("(fetch failed)")

FIREBASE API: STORY (id=22175525)
{
  "by": "fortran77",
  "descendants": 0,
  "id": 22175525,
  "score": 3,
  "time": 1580255543,
  "title": "How To turn off Ultra Wideband U1 to stop background location tracking on iPhone",
  "type": "story",
  "url": "https://9to5mac.com/2020/01/28/how-to-turn-off-ultra-wideband-u1-chip-prevent-background-location-tracking-iphone-11/"
}

FIREBASE API: COMMENT (id=22197310)
{
  "by": "StreamBright",
  "id": 22197310,
  "kids": [
    22197433
  ],
  "parent": 22197115,
  "text": "iPhone is ARM. Google cross compiles most of its code in the last X years. Windows 10 runs on ARM. Not sure why you think it would be a problem.",
  "time": 1580428125,
  "type": "comment"
}


In [8]:
# Check if mapping between Algolia and Firebase is correct
idx = 0
row = df_hn.iloc[idx]
oid = row["object_id"]
ag_text = row["text"][:200] if row["text"] else ""
ag_author = row["author"]
ag_item_type = row["item_type"]

fb = hn_firebase_get(oid)

if fb:
    print("ALGOLIA AND FIREBASE MAPPING CHECK\n")
    print("object_id:", oid)
    print("\nAlgolia API:")
    print("author:", ag_author, "  item_type:", ag_item_type)
    print("text:", ag_text[:150] + "..." if len(ag_text) > 150 else ag_text)
    print("\nFirebase API:")
    print("by:", fb.get("by"), "  type:", fb.get("type"))
    fb_text = (fb.get("text") or "")[:200]
    print("text: ", fb_text[:150] + "..." if len(fb_text) > 150 else fb_text)
    print("\nAuthor/type match:", ag_author == fb.get("by") and ag_item_type == str(fb.get("type", "")))
    print("ID match:", str(oid) == str(fb.get("id", "")))
else:
    print("Firebase fetch failed for object_id:", oid)

ALGOLIA AND FIREBASE MAPPING CHECK

object_id: 22197310

Algolia API:
author: StreamBright   item_type: comment
text: iPhone is ARM. Google cross compiles most of its code in the last X years. Windows 10 runs on ARM. Not sure why you think it would be a problem.

Firebase API:
by: StreamBright   type: comment
text:  iPhone is ARM. Google cross compiles most of its code in the last X years. Windows 10 runs on ARM. Not sure why you think it would be a problem.

Author/type match: True
ID match: True


In [ ]:
# Basic structure cleanup (used in this section)
def finalize_hn(df_hn: pd.DataFrame) -> pd.DataFrame:
    df_all = df_hn.copy()
    required = ["doc_id","title","query","created_at","author","tags","source_url","text"]
    optional = ["deleted","descendants","score","item_type","object_id"]
    for c in optional:
        if c not in df_all.columns:
            df_all[c] = None if c == "object_id" else (False if c == "deleted" else ("" if c == "item_type" else 0))
    for c in required:
        if c not in df_all.columns:
            raise ValueError(f"Missing required column '{c}'.")
    return df_all.reset_index(drop=True)

df_all = finalize_hn(df_hn)
print("df_all rows:", len(df_all))

## 5. Data Cleaning & Export

In [20]:
# -- Cleaning & Filtering --
# Determine if a given text is low quality, e.g. too short or mostly URLs
def is_low_quality_text(text: str, min_words: int = 5, url_ratio: float = 0.5) -> bool:
    s = (text or "").strip()
    words = s.split()
    if len(words) < min_words:
        return True
    url_chars = sum(len(m) for m in re.findall(r"https?://\S+|www\.\S+", s))
    return url_chars / max(len(s), 1) > url_ratio

_ILLEGAL = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")

# Remove illegal Excel control chars from strings
def clean_excel_cell(x):
    if x is None:
        return ""
    if isinstance(x, (int, float, bool)):
        return x
    s = str(x)
    return _ILLEGAL.sub("", s)

# Apply clean_excel_cell to all object/string columns
def make_excel_safe(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    for c in df2.columns:
        if df2[c].dtype == "object":
            df2[c] = df2[c].map(clean_excel_cell)
    return df2

# Remove HN quotes from text
def remove_hn_quotes(text: str) -> str:
    if not text or not isinstance(text, str):
        return text
    lines = text.splitlines()
    out = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith(">"):
            stripped = stripped[1:].strip()  # remove leading >
        if stripped:
            out.append(stripped)
    return " ".join(out) if out else ""

In [ ]:
# Perform data cleaning steps
df_corpus = df_all.copy()

# 1. Text cleanup: cast to str, strip, unicode normalize, drop empty
df_corpus["text"] = df_corpus["text"].astype(str).str.strip()
df_corpus["text"] = df_corpus["text"].apply(
    # Use Unicode normalization form NFKC to convert characters to their canonical forms
    # e.g. "１ＡＢＣ" (full-width digits/letters) becomes "1ABC"
    lambda s: unicodedata.normalize("NFKC", s) if isinstance(s, str) else s
)
df_corpus = df_corpus[df_corpus["text"].str.len() > 0]
n_after_empty = len(df_corpus)

# 2. Remove HN quotes from text
df_corpus["text"] = df_corpus["text"].apply(remove_hn_quotes)

# 3. Quality filters: drop low-quality text, empty author
n_before_quality = len(df_corpus)
low_quality_mask = df_corpus["text"].apply(is_low_quality_text)
n_low_quality = low_quality_mask.sum()
# Print sample of low-quality texts
low_quality_df = df_corpus[low_quality_mask]
if len(low_quality_df) > 0:
    n_show = min(5, len(low_quality_df))
    print(f"Sample of low-quality texts (total filtered: {n_low_quality}):\n")
    for idx, row in low_quality_df.head(n_show).iterrows():
        t = str(row["text"] or "")
        words = len(t.split())
        url_chars = sum(len(m) for m in re.findall(r"https?://\S+|www\.\S+", t))
        ratio = url_chars / max(len(t), 1)
        preview = (t[:120] + "...") if len(t) > 120 else t
        print(f"[{words} words, url_ratio={ratio:.2f}] {preview}\n")
else:
    print("No low-quality texts found.")
df_corpus = df_corpus[~low_quality_mask]

if "author" in df_corpus.columns:
    n_before_author = len(df_corpus)
    df_corpus = df_corpus[df_corpus["author"].astype(str).str.strip().str.len() > 0]
    n_empty_author = n_before_author - len(df_corpus)
else:
    n_empty_author = 0

if "deleted" in df_corpus.columns:
    n_before_del = len(df_corpus)
    df_corpus = df_corpus[~df_corpus["deleted"].fillna(False).astype(bool)]
    n_deleted = n_before_del - len(df_corpus)
else:
    n_deleted = 0

print(f"Low quality: {n_low_quality} | Empty author: {n_empty_author} | Deleted: {n_deleted} | Rows: {n_before_quality} -> {len(df_corpus)}")

# 4. Deduplicate: one row per doc_id, then one per unique text
n_before_dedup = len(df_corpus)
if "doc_id" in df_corpus.columns:
    df_corpus = df_corpus.drop_duplicates(subset=["doc_id"])
    n_after_doc_id = len(df_corpus)
    df_corpus = df_corpus.drop_duplicates(subset=["text"]).reset_index(drop=True)
    n_after_text_dedup = len(df_corpus)
    print(f"Doc_id duplicates removed: {n_before_dedup - n_after_doc_id} | Text duplicates removed: {n_after_doc_id - n_after_text_dedup} | Rows: {n_before_dedup} -> {n_after_text_dedup}")
else:
    df_corpus = df_corpus.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"Text duplicates removed: {n_before_dedup - len(df_corpus)} | Rows: {n_before_dedup} -> {len(df_corpus)}")

# 5. Remove illegal Excel chars before export
df_corpus = make_excel_safe(df_corpus)

Sample of low-quality texts (total filtered: 92):

[5 words, url_ratio=0.72] https://support.apple.com/iphone/repair/service/battery-powe... warned that they'll try.

[12 words, url_ratio=0.55] This is one of the reasons I jailbreak my iPhone =p https://repo.packix.com/package/com.alwayslatesttimelinetwit...

[24 words, url_ratio=0.56] Already exists (well, not iPhone, but...) https://www.samsung.com/us/business/solutions/industries/gov... And accessorie...

[12 words, url_ratio=0.55] According to vice [1] he was using an iPhone X. 1: https://www.vice.com/en_us/article/v74v34/saudi-arabia-hacke...

[9 words, url_ratio=0.63] I can see the excerpt too. Try this: https://duckduckgo.com/?q=centos+7+tuned+no+daemon&t=iphone&...

Low quality: 92 | Empty author: 0 | Deleted: 0 | Rows: 18726 -> 18634
Doc_id duplicates removed: 0 | Text duplicates removed: 0 | Rows: 18634 -> 18634


In [23]:
# Save outputs
df_corpus.to_excel("corpus_full.xlsx", index=False, engine="openpyxl")
print("Saved corpus_full.xlsx | rows:", len(df_corpus), "| cols:", len(df_corpus.columns))

# Eval file: text + empty label column for manual annotation
df_eval = df_corpus[["text"]].copy()
df_eval["label"] = ""
df_eval = make_excel_safe(df_eval)
df_eval.to_excel("eval.xlsx", index=False, engine="openpyxl")
print("Saved eval.xlsx | rows:", len(df_eval))

Saved corpus_full.xlsx | rows: 18634 | cols: 13
Saved eval.xlsx | rows: 18634
